# Exercise Contextual versus BoW embeddings
----

In this exercise, we will use a synthetic dataset: data created by AI for illustration purposes, you may additionally also try using your own data.



In [ ]:
#PACKAGES -
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np



### 1. Read in the data

In [ ]:
#here is a list of sentences that show some variation in the meaning of the words "bank", "cool", "trunk", "duck", and "figure" based on context. The sentences are repeated to show that the same sentence can have different meanings based on context.
sentences = ["I'm going to the bank.", "I'm going to the bank.", "I'm going to the bank.", "I need to visit the bank today.", "I'm heading to the bank to withdraw some cash.", "The bank is where I'm off to.", "I have an appointment at the bank.", "Let's spend the day by the river bank.", "The river bank is a peaceful place to relax.", "I enjoy walking along the river bank.", "We can have a picnic by the river bank.",
"She has a cool job.", "She has a cool job.", "She has a cool job.", "Her job is really interesting and fun.", "She works in a creative field and loves it.", "That job of hers is so unique.", "She's lucky to have such a cool profession.", "Her workplace is always chilly.", "She works in a refrigerated environment.", "The temperature in her office is freezing.", "She needs to bundle up for her job.",
"You need to check the trunk.", "You need to check the trunk.", "You need to check the trunk.", "Don't forget to look in the trunk of the car.", "There might be something important in the trunk.", "Make sure to verify the contents of the trunk.", "The trunk needs to be inspected for any damage.", "Take a look at the tree trunk for any damage.", "The trunk of the old tree might have some interesting carvings.", "Check if the tree trunk needs to be treated for pests.", "See if there are any unique patterns or textures on the tree trunk.",
"I saw her duck.", "I saw her duck.", "I saw her duck.", "Her duck was waddling in the park.", "The duck she owns is so cute.", "I spotted her duck by the pond.", "Her duck was quacking loudly.", "She ducked to avoid the flying object.", "Her quick ducking saved her from the falling branch.", "I noticed her sudden ducking movement.", "She ducked and dodged the incoming ball.",
"She has a great figure.", "She has a great figure.", "She has a great figure.", "Her body shape is very flattering.", "She carries herself with grace and confidence.", "Her figure is well-proportioned and attractive.", "She knows how to dress to highlight her figure.", "The numerical figure she presented was impressive.", "Her calculations yielded a significant figure.", "The data supports a substantial figure.", "The figure she quoted was accurate and reliable."]

#these are the contexts for the sentences above, which show that the same sentence can have different meanings based on context. Note that some sentences can have multiple contexts, which is why they are repeated in the list.
context = ["bank", "bank-money", "bank-river", "bank-money", "bank-money", "bank-money", "bank-money", "bank-river", "bank-river", "bank-river", "bank-river", "cool", "cool-nice", "cool-cold", "cool-nice", "cool-nice", "cool-nice", "cool-nice", "cool-cold", "cool-cold", "cool-cold", "cool-cold", "trunk", "trunk-car", "trunk-tree", "trunk-car", "trunk-car", "trunk-car", "trunk-car", "trunk-tree", "trunk-tree", "trunk-tree", "trunk-tree", 
           "duck", "duck-animal", "duck-down", "duck-animal", "duck-animal", "duck-animal", "duck-animal", "duck-down", "duck-down", "duck-down", "duck-down", "figure", "figure-body", "figure-number", "figure-body", "figure-body", "figure-body", "figure-body", "figure-number", "figure-number", "figure-number", "figure-number"]

#these are the set labels for the sentences above, which help us select helpful examples.
set_labels = ["bank-base", "bank-base-money", "bank-base-river", "bank-money", "bank-money", "bank-money", "bank-money", "bank-river", "bank-river", "bank-river", "bank-river", "cool-base", "cool-base-nice", "cool-base-cold", "cool-nice", "cool-nice", "cool-nice", "cool-nice", "cool-cold", "cool-cold", "cool-cold", "cool-cold", "trunk-base", "trunk-base-car", "trunk-base-tree", "trunk-car", "trunk-car", "trunk-car", "trunk-car", "trunk-tree", "trunk-tree", "trunk-tree", "trunk-tree", 
           "duck-base", "duck-base-animal", "duck-base-down", "duck-animal", "duck-animal", "duck-animal", "duck-animal", "duck-down", "duck-down", "duck-down", "duck-down", "figure-base", "figure-base-body", "figure-base-number", "figure-body", "figure-body", "figure-body", "figure-body", "figure-number", "figure-number", "figure-number", "figure-number"]

#now we can create a pandas dataframe to hold the sentences, their contexts, and their set labels. This will make it easier to work with the data and select examples for our analysis.
contextdf = pd.DataFrame({'Sentence': sentences, 'Context': context, 'Set': set_labels})

We will use the figure subsets of this dataset: 
1. Sentences referring to *figure* (body vs. number)


In [ ]:
#collect the set 
figure_df = contextdf[contextdf['Set'].isin(['figure-body', 'figure-number'])].reset_index(drop=True).copy()
print(figure_df)


### 2. The BoW baseline - Vectorization

<img src="../../images/03_vectorization.png" alt="Preprocessing diagram" style="max-width: 150px;">

In [ ]:
#same count vectorizer as day1 for reference
from sklearn.feature_extraction.text import CountVectorizer #a vectorizer (converts text to numbers)

def my_tokenizer(text):
    return text.split()  # just split the text into words based on whitespace. This is a simple tokenizer that does not perform any additional preprocessing.

vectorizer_CV_raw = CountVectorizer(tokenizer=my_tokenizer, lowercase=False, token_pattern=None) # Note: we start with the original text for comparison.

# Fit the vectorizer to the text, and transform the text to a document-term matrix
# Let's just use four sentences for this example, to keep the output manageable, but include the different uses of the word "figure" in context. We will use the sentences at index 2, 3, 4, and 5 in the figure_df dataframe.
# First print the sentences we are using for this example:
print("Sentences used for this example:")
print(figure_df['Sentence'][2:6].to_list())

# Now fit the vectorizer to the text, and transform the text to a document-term matrix.
X_CV = vectorizer_CV_raw.fit_transform(figure_df['Sentence'][2:6].to_list())

#print the number of terms (i.e., columns) in the document-term matrix
print(f"CountVectorizer - number of features: {len(vectorizer_CV_raw.get_feature_names_out())}")

#print the number of documents (i.e., rows) in the document-term matrix:
print(f"CountVectorizer - number of documents: {X_CV.shape[0]}")

#inspect the document-term matrix as a DataFrame for better readability - This is what the computer gets as input.
X_CV_df_raw = pd.DataFrame(X_CV.toarray(), columns=vectorizer_CV_raw.get_feature_names_out())
print("\nCountVectorizer - document-term matrix:\n")
print(X_CV_df_raw)

#Let's list the included features (i.e., words) in the vocabulary:
print("\nCountVectorizer - features (vocabulary):\n")
print(vectorizer_CV_raw.get_feature_names_out())

#Inspect the document-term matrix as a DataFrame for better readability - This is what the computer gets as input.
X_CV_df_raw = pd.DataFrame(X_CV.toarray(), columns=vectorizer_CV_raw.get_feature_names_out()) 
# The row numbers represent the sentences, the column labels represent the words in the vocabulary. 
# The values in the matrix represent the frequency of the word in the sentence.
# Note: We use sentences here, but the same applies to documents in general, including complete newspapers if you want to.
print("\nCountVectorizer - document-term matrix:\n")
print(X_CV_df_raw)

Now try contextual embeddings



In [ ]:
# What changes if we use contextual embeddings instead of a bag-of-words approach? Let's try it out and see what happens.
# We use the same transformer as yesterday, which is a pre-trained model that can generate embeddings for sentences. The embeddings are numerical representations of the sentences that capture their semantic meaning. 
# This model is especially useful for comparing sentences that have the same words but different meanings based on context, as we have in our example with the word "figure".

# Define an embedding model to vectorize our documents:
# NOTE: cuda > cpu, or use "mps" for Apple Silicon
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")


#compute the embeddings (this may take a while on a CPU! -> that's why we only use four sentences here, and also to keep the output manageable):
embeddings = embedding_model.encode(figure_df['Sentence'][2:6].to_list(), show_progress_bar=True)


In [ ]:
# What do these embeddings look like? 
# Let's inspect the shape of the embeddings and the first few values of the first embedding vector.
print(f"Shape of embeddings: {embeddings.shape}")
print(f"First 10 values of each sentence embedding vector: {embeddings[:2,:10]}")


We now have 384 features with each a non-integer value, even though our 4 sentence corpus includes far fewer words, yielding only 24 BoW features.

Note how we embed sentences here, rather than words!

Question: What do you think would happen with the number of features of our embedding model and BoW vectorizor if we used a larger corpus?



In [ ]:
# Illustration: Use the embeddings to compare the sentences

# Since we have the embeddings, we can now use them to compare the sentences. Let's calculate the cosine similarity between the four sentences in our example.
# Calculate pairwise similarity between the four sentences

for i in range(len(embeddings)):
    for j in range(i + 1, len(embeddings)):
        similarity = cosine_similarity([embeddings[i]], [embeddings[j]])[0][0]
        print(f"Similarity between sentence {i+2} and sentence {j+2}: {similarity:.4f}")  # Values between -1 and 1, since the embeddings are ordered per sentence 0 to 3, but refer to sentences 2 to 5 in the original dataframe, we can use i+1 and j+1 to represent the sentence numbers in the output.



Did you expect this result?

In [ ]:
#Can we get the same result with our BoW document term matrix? Let's try it out and see what happens. We will use the same four sentences as before, and calculate the cosine similarity between them using the document-term matrix we created earlier.

print("Document-Term Matrix:")
print(X_CV.toarray())

# Calculate cosine similarity
for i in range(X_CV.shape[0]):
    for j in range(i + 1, X_CV.shape[0]):
        similarity_bow = cosine_similarity(X_CV[i:i+1], X_CV[j:j+1])[0][0]
        print(f"BoW Cosine Similarity between sentence {i+2} and sentence {j+2}: {similarity_bow:.4f}")  # Values between 0 and 1, since the BoW representation is non-negative. The sentences are ordered per sentence 1 to 4, but refer to sentences 2 to 5 in the original dataframe, we can use i+1 and j+1 to represent the sentence numbers in the output.


Why do you think the Bag of Words vectors did not show the similarity and difference between our sentences?

What if we used a different transformer (the one we will use in our prediction task later this afternoon)?
This model ("mmBERT-small") is multilingual, which is nice, but not developed for text classification rather than comparing sentence similarity

In [ ]:
# Load mmBERT-small
model_name = "jhu-clsp/mmBERT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)


# ===== mmBERT Embeddings =====
# Note: no easy sentence transformer pipeline available for this model, so we will use the tokenizer and model directly to get the embeddings. 
# We will use mean pooling to get sentence embeddings from the last hidden state of the model.
inputs = tokenizer(figure_df['Sentence'][2:6].to_list(), padding=True, truncation=True, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    # Use mean pooling to get sentence embeddings
    embeddings = outputs.last_hidden_state.mean(dim=1)

# Calculate pairwise similarity between the four sentences
for i in range(len(embeddings)):
    for j in range(i + 1, len(embeddings)):
        mmbert_similarity = cosine_similarity([embeddings[i].numpy()], [embeddings[j].numpy()])[0][0]
        print(f"mmBERT Similarity between sentence {i+2} and sentence {j+2}: {mmbert_similarity:.4f}")

# ===== Show the embeddings =====
print(f"mmBERT vector shape: {embeddings[0].shape}")   # (384,)

print(f"\nFirst sentence mmBERT (first 10 dims):\n{embeddings[0][:10].numpy()}")

What do you think of this result?  

What do you think could be the reason these sentences are so similar to this model?
 